[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/02_memory_coalescing/02_memory_coalescing.ipynb)

# 02 · 内存层级与合并访问（用 numpy 模拟）

目标：把 **合并访问（coalescing）**、**行优先布局对访存的影响**、**shared memory 复用**、**bank 冲突与 padding**、**分块矩阵转置**、**算子融合省 HBM** 全部用 numpy 模拟出来，并用 `assert` 验证。

路线：cache-line 事务模型 → 行优先按行/按列访问 → bank 冲突模型 → 分块转置对拍 `A.T` → 融合 vs 未融合 HBM 流量 → ✏️ 练习 → 📖 答案 → 🧪 真实 GPU 带宽胶囊。

> 心智模型：**HBM 一次搬一整条 cache line；一个 warp 的 32 个地址落在几条 line 上，就是几次事务**。我们不测真实时延，只精确建模**访问模式**——这才是访存性能的根。

## 1 · cache-line 事务模型：合并 vs 非合并

硬件以 **cache line**（这里取 128 字节）为最小搬运单位。一个 warp（32 线程）同时取数时，**事务数 = 这 32 个字节地址覆盖的不同 cache line 数**。

- **合并**：32 个连续 float（`i*4`）→ 字节 0..127 → 全在 1 条 128B line → **1 次事务**。
- **非合并（跨步 128B）**：每个 float 各占一条 line → **32 次事务**，每条 line 只用了 4/128 = 3%。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
WARP = 32
LINE = 128            # cache line 字节数
FP32 = 4              # 一个 float32 的字节数

def count_transactions(addresses_bytes, line_bytes=LINE):
    '''一个 warp 的若干字节地址 -> 覆盖的不同 cache line 数（=内存事务数）。'''
    return len({a // line_bytes for a in addresses_bytes})

def bandwidth_efficiency(addresses_bytes, useful_bytes_each=FP32, line_bytes=LINE):
    '''真正要用的字节 / 实际搬运的字节。'''
    tx = count_transactions(addresses_bytes, line_bytes)
    useful = len(addresses_bytes) * useful_bytes_each
    moved = tx * line_bytes
    return useful / moved

coalesced = [i * FP32 for i in range(WARP)]        # 0,4,8,...,124
strided   = [i * LINE for i in range(WARP)]        # 0,128,256,...

tx_c = count_transactions(coalesced)
tx_s = count_transactions(strided)
print(f'合并访问  : {tx_c:2d} 次事务, 带宽有效率 {bandwidth_efficiency(coalesced):6.1%}')
print(f'非合并访问: {tx_s:2d} 次事务, 带宽有效率 {bandwidth_efficiency(strided):6.1%}')
assert tx_c == 1, '32 个连续 float 应只需 1 次事务'
assert tx_s == 32, '跨步 128B 的 32 次访问应需 32 次事务'
assert bandwidth_efficiency(strided) < 0.05
print('✅ 同样取 32 个 float，非合并多花 32 倍带宽 —— 全浪费在用不上的字节上')

## 2 · 行优先布局：按行访问合并、按列访问跨步

矩阵 `M[r][c]` 行优先时线性下标 `r*ncols + c`，字节地址 `(r*ncols+c)*4`。

一个 warp 若读**一行的 32 列**（相邻线程→相邻列）→ 地址连续 → 合并；
若读**一列的 32 行**（相邻线程→相邻行）→ 地址步长 `ncols*4` → 跨步、非合并。

In [ ]:
def addr(r, c, ncols, dtype_bytes=FP32):
    return (r * ncols + c) * dtype_bytes

ncols = 1024
# warp 读 第 r 行 的 32 个连续列
row_access = [addr(5, c, ncols) for c in range(WARP)]
# warp 读 第 c 列 的 32 个连续行
col_access = [addr(r, 7, ncols) for r in range(WARP)]

print(f'按行访问 M[5][0..31] : {count_transactions(row_access):2d} 次事务 (步长 {row_access[1]-row_access[0]} 字节)')
print(f'按列访问 M[0..31][7] : {count_transactions(col_access):2d} 次事务 (步长 {col_access[1]-col_access[0]} 字节)')
assert count_transactions(row_access) == 1, '按行应合并'
assert count_transactions(col_access) == WARP, '按列每个元素独占一条 line'
print('✅ 同一矩阵，按行 1 次事务、按列 32 次 —— 线程映射必须顺着内存布局走')

## 3 · shared memory 的 bank 冲突与 padding

shared memory 有 **32 个 bank**，第 `w` 个 4 字节字落在 `bank = w % 32`。
一个 warp 若多个线程访问**同一 bank 的不同地址**，就串行化（k-way conflict，慢 k 倍）。

经典翻车：32 宽 tile 读**一整列**。`tile[t][c]` 的字索引 `t*32+c` → `bank=(t*32+c)%32=c`，对所有行 `t` 都是同一个 bank → **32-way 冲突**。补一列变 33 宽 → `bank=(t*33+c)%32=(t+c)%32` → 全不同 → **0 冲突**。

In [ ]:
from collections import Counter
N_BANKS = 32

def bank_of(word_index, n_banks=N_BANKS):
    return word_index % n_banks

def max_bank_conflict(word_indices, n_banks=N_BANKS):
    '''一个 warp 各线程访问的字索引 -> 最拥挤 bank 的访问数（=串行倍数, 1 表示无冲突）。'''
    banks = [bank_of(w, n_banks) for w in word_indices]
    return max(Counter(banks).values())

col = 7
# 32 宽 tile，warp 读列 col 的 32 行：字索引 t*32+col
words_unpadded = [t * 32 + col for t in range(WARP)]
# 33 宽 tile（padding 一列）：字索引 t*33+col
words_padded   = [t * 33 + col for t in range(WARP)]

print('未 padding (32 宽) 列访问  banks =', [bank_of(w) for w in words_unpadded][:8], '...')
print('  -> 最大冲突度 =', max_bank_conflict(words_unpadded), '(32-way, 串行 32 次)')
print('已 padding (33 宽) 列访问  banks =', [bank_of(w) for w in words_padded][:8], '...')
print('  -> 最大冲突度 =', max_bank_conflict(words_padded), '(无冲突, 一拍完成)')
assert max_bank_conflict(words_unpadded) == 32
assert max_bank_conflict(words_padded) == 1
print('✅ 列数 32 -> 33 的一个 padding，把 32-way 冲突降为 0')

## 4 · 分块矩阵转置：让读写双双合并

朴素转置 `B[c][r]=A[r][c]`：读 A 合并则写 B 跨步（或反之），必有一端非合并。

**分块转置**借道 shared memory：把 A 的一个 tile **合并读**进 `tile_buf`，片上转置，再**合并写**回 B 的对应 tile。
下面用 numpy 模拟这个**数据搬运过程**（`tile_buf` 就是 shared memory），并对拍 `A.T` 验证正确。

In [ ]:
def transpose_tiled(A, tile=32):
    '''模拟 GPU 分块转置：逐 tile 搬进 shared(tile_buf)、片上转置、写回。
       含边界处理（矩阵尺寸不是 tile 整数倍）。结果应等于 A.T。'''
    H, W = A.shape
    B = np.empty((W, H), dtype=A.dtype)
    for i in range(0, H, tile):
        for j in range(0, W, tile):
            ii = min(i + tile, H)
            jj = min(j + tile, W)
            tile_buf = A[i:ii, j:jj]          # 合并读：A 的一块(按行连续)
            B[j:jj, i:ii] = tile_buf.T        # 片上转置后，合并写：B 的对应块
    return B

A = rng.standard_normal((64, 64))
B = transpose_tiled(A, tile=32)
assert B.shape == (64, 64)
assert np.allclose(B, A.T, atol=1e-12), '分块转置必须逐位等于 A.T'
print('✅ 64x64 分块转置对拍 A.T 通过')

# 非方阵 + 非整除尺寸，验证边界处理
A2 = rng.standard_normal((70, 50))
assert np.allclose(transpose_tiled(A2, tile=16), A2.T, atol=1e-12)
print('✅ 70x50 (tile=16, 含边界) 分块转置对拍 A.T 通过')

## 5 · 算子融合：少跑几趟 HBM

逐元素链 `y = relu(a*x + b)`。未融合 = 多个内核，每个都完整读+写一遍 HBM；融合 = 一个内核，中间结果留寄存器，只读一次 x、写一次 y。

我们建模 **HBM 字节流量**（fp32，每元素 4 字节），看融合省了多少。

In [ ]:
def hbm_bytes_unfused(n, n_ops, dtype_bytes=FP32):
    '''n_ops 个内核，每个读 n + 写 n 个元素。'''
    return n_ops * (dtype_bytes * n + dtype_bytes * n)

def hbm_bytes_fused(n, dtype_bytes=FP32):
    '''一个融合内核：读一次 x + 写一次 y。'''
    return dtype_bytes * n + dtype_bytes * n

n = 1_000_000
n_ops = 3                                  # 乘、加、relu
un = hbm_bytes_unfused(n, n_ops)
fu = hbm_bytes_fused(n)
print(f'未融合 ({n_ops} 个内核): {un/1e6:6.1f} MB HBM 流量')
print(f'融合   (1 个内核)   : {fu/1e6:6.1f} MB HBM 流量')
print(f'省下 {1 - fu/un:.0%} 的带宽')
assert fu * n_ops == un, '融合应把流量从 n_ops 趟降到 1 趟'
assert fu < un
print('✅ 逐元素算子融合：HBM 流量 6n -> 2n，这是模块 04/05 的预告')

## 6 · 小结模拟：把三种访问模式排个队

用前面的工具，把「合并 / 跨步 / 随机」三种 warp 访问模式的事务数与带宽有效率并排打出来，直观感受访存模式对带宽的统治力。

In [ ]:
patterns = {
    '合并(连续)':   [i * FP32 for i in range(WARP)],
    '跨步(128B)':   [i * LINE for i in range(WARP)],
    '随机散列':     sorted(rng.integers(0, 1_000_000, size=WARP).tolist()),
}
print(f"{'模式':<12s}{'事务数':>8s}{'带宽有效率':>12s}")
for name, addrs in patterns.items():
    tx = count_transactions(addrs)
    eff = bandwidth_efficiency(addrs)
    print(f'{name:<12s}{tx:>8d}{eff:>11.1%}')
assert count_transactions(patterns['合并(连续)']) == 1
assert count_transactions(patterns['跨步(128B)']) == WARP
print('\n✅ 结论：合并访问是访存受限内核的第一优化目标 —— 它直接决定有效带宽')

---
## ✏️ 练习 1：合并度量

从零实现 `count_transactions(addresses, line_bytes)` 与 `coalescing_efficiency(addresses, useful_each, line_bytes)`，并用它们区分合并与跨步访问。

（提示：事务数 = 不同的 `addr // line_bytes` 个数；有效率 = 有用字节 / (事务数 × line_bytes)。）

In [ ]:
def count_transactions(addresses_bytes, line_bytes=128):
    # TODO: 返回这些字节地址覆盖的不同 cache line 数
    raise NotImplementedError

def coalescing_efficiency(addresses_bytes, useful_each=4, line_bytes=128):
    # TODO: 返回 有用字节 / 实际搬运字节
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
good = [i * 4 for i in range(32)]            # 合并
bad  = [i * 128 for i in range(32)]          # 跨步
assert count_transactions(good) == 1
assert count_transactions(bad) == 32
assert abs(coalescing_efficiency(good) - 1.0) < 1e-9
assert coalescing_efficiency(bad) < 0.05
# 半合并：每 2 个 float 跨一条 line
half = [i * 64 for i in range(32)]           # 步长 64B，2 个/line? 不，64B 仍各自... 验证一致性
assert count_transactions(half) == len({a // 128 for a in half})
print('✅ 练习 1 通过：能量化任意访问模式的合并度')

## ✏️ 练习 2：bank 冲突与 padding

实现 `bank_of(word_index, n_banks=32)` 与 `max_bank_conflict(word_indices)`，验证一个 `width` 宽 shared tile 读整列时的冲突度，并证明把宽度改成 `width+1` 能消除冲突。

实现 `column_access_words(width, col, n_rows=32)`：返回读第 `col` 列、`n_rows` 行的字索引列表。

In [ ]:
from collections import Counter

def bank_of(word_index, n_banks=32):
    # TODO
    raise NotImplementedError

def max_bank_conflict(word_indices, n_banks=32):
    # TODO: 最拥挤 bank 的访问数
    raise NotImplementedError

def column_access_words(width, col, n_rows=32):
    # TODO: tile 宽 width，读第 col 列、第 0..n_rows-1 行 -> 字索引 row*width+col
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert bank_of(0) == 0 and bank_of(33) == 1 and bank_of(64) == 0
w32 = column_access_words(32, col=3)
w33 = column_access_words(33, col=3)
assert max_bank_conflict(w32) == 32, '32 宽 tile 读整列 -> 32-way 冲突'
assert max_bank_conflict(w33) == 1,  'padding 到 33 宽 -> 无冲突'
# 对任意列都成立
for col in range(5):
    assert max_bank_conflict(column_access_words(33, col)) == 1
print('✅ 练习 2 通过：padding +1 是消 bank 冲突的通用招')

## ✏️ 练习 3：分块转置（含边界）

实现 `transpose_tiled(A, tile)`：逐 tile 借道 `tile_buf`（模拟 shared memory）完成 `A.T`，**必须正确处理非整除尺寸**。对一个 70×50 矩阵、`tile=16` 验证逐位等于 `A.T`。

In [ ]:
def transpose_tiled(A, tile=32):
    H, W = A.shape
    B = np.empty((W, H), dtype=A.dtype)
    # TODO: 双重循环遍历 tile，处理边界 (min(i+tile,H) 等)，
    #       tile_buf = A[i:ii, j:jj]; B[j:jj, i:ii] = tile_buf.T
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for (H, W, t) in [(64, 64, 32), (70, 50, 16), (33, 17, 8), (100, 1, 16)]:
    M = rng.standard_normal((H, W))
    out = transpose_tiled(M, tile=t)
    assert out.shape == (W, H), f'转置后形状应为 {(W, H)}'
    assert np.allclose(out, M.T, atol=1e-12), f'{(H,W,t)} 转置错误'
print('✅ 练习 3 通过：分块转置在各种尺寸/边界下都等于 A.T')

## ✏️ 练习 4：融合省了多少带宽？

实现 `hbm_savings(n, n_ops, dtype_bytes)`：返回把 `n_ops` 个逐元素内核融合成 1 个后，**节省的 HBM 字节数** 与 **节省比例**（元组）。

In [ ]:
def hbm_savings(n, n_ops, dtype_bytes=4):
    # TODO: 未融合 = n_ops*(读n+写n)*bytes; 融合 = (读n+写n)*bytes
    #       返回 (节省字节, 节省比例)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
saved_bytes, saved_frac = hbm_savings(1_000_000, n_ops=3)
assert abs(saved_frac - 2/3) < 1e-9, '3 个内核融合应省 2/3 流量'
assert saved_bytes == 2 * (2 * 4 * 1_000_000), '省下的是 (n_ops-1) 趟读写'
# n_ops=1 时无可省
sb, sf = hbm_savings(1000, n_ops=1)
assert sb == 0 and abs(sf) < 1e-12
print('✅ 练习 4 通过：融合 n_ops 个内核省下 (n_ops-1)/n_ops 的 HBM 流量')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def count_transactions(addresses_bytes, line_bytes=128):
    return len({a // line_bytes for a in addresses_bytes})

def coalescing_efficiency(addresses_bytes, useful_each=4, line_bytes=128):
    tx = count_transactions(addresses_bytes, line_bytes)
    useful = len(addresses_bytes) * useful_each
    return useful / (tx * line_bytes)

In [ ]:
# 练习 2 参考答案
from collections import Counter
def bank_of(word_index, n_banks=32):
    return word_index % n_banks

def max_bank_conflict(word_indices, n_banks=32):
    banks = [w % n_banks for w in word_indices]
    return max(Counter(banks).values())

def column_access_words(width, col, n_rows=32):
    return [row * width + col for row in range(n_rows)]

In [ ]:
# 练习 3 参考答案
def transpose_tiled(A, tile=32):
    H, W = A.shape
    B = np.empty((W, H), dtype=A.dtype)
    for i in range(0, H, tile):
        for j in range(0, W, tile):
            ii, jj = min(i + tile, H), min(j + tile, W)
            tile_buf = A[i:ii, j:jj]
            B[j:jj, i:ii] = tile_buf.T
    return B

In [ ]:
# 练习 4 参考答案
def hbm_savings(n, n_ops, dtype_bytes=4):
    unfused = n_ops * (2 * dtype_bytes * n)
    fused = 2 * dtype_bytes * n
    saved = unfused - fused
    return saved, saved / unfused

---
## 🧪 真实数据胶囊：用真实 HBM 带宽算一笔账

用真实 GPU 的 HBM 带宽，算两件事：① 把一个 (4096, 4096) fp16 矩阵从 HBM 流式读一遍要多久（**下界**，假设带宽 100% 用满）；② 一个 fp16 矩阵乘 `C=A·B` 的算术强度，判断它该是访存受限还是算力受限。

In [ ]:
# 真实 GPU HBM 带宽（公开规格, 约数）
HBM = {
    'A100': dict(tbps=2.039, tflops=312.0),   # FP16 张量核心
    'H100': dict(tbps=3.35,  tflops=989.0),
}
FP16 = 2

def min_read_time(num_bytes, tbps):
    '''流式读 num_bytes 的时间下界(秒) = 字节 / 带宽(字节每秒)。'''
    return num_bytes / (tbps * 1e12)

n = 4096
mat_bytes = n * n * FP16
print(f'(4096x4096) fp16 矩阵 = {mat_bytes/1e6:.1f} MB')
for name, s in HBM.items():
    t = min_read_time(mat_bytes, s['tbps'])
    print(f'  {name}: 流式读一遍 >= {t*1e6:7.1f} us  (带宽 {s["tbps"]} TB/s)')

# 矩阵乘 C=A@B 的算术强度 (n^3 次 FMA = 2n^3 FLOPs; 读 A,B 写 C = 3 n^2 个 fp16)
flops = 2 * n**3
bytes_moved = 3 * n * n * FP16
ai = flops / bytes_moved
print(f'\nGEMM 算术强度 = {ai:.0f} FLOP/byte')
for name, s in HBM.items():
    ridge = s['tflops'] * 1e12 / (s['tbps'] * 1e12)
    verdict = '算力受限' if ai > ridge else '访存受限'
    print(f'  {name}: 脊点 {ridge:.0f} FLOP/byte -> 该 GEMM 是 {verdict}')
assert min_read_time(mat_bytes, 3.35) < min_read_time(mat_bytes, 2.039)
print(f'\n观察：大 GEMM 算术强度高(~{ai:.0f} FLOP/byte)，远在脊点右侧 -> 算力受限(模块03会把它做到)；'
      '\n但逐元素/softmax 算术强度~O(1) -> 访存受限 -> 靠合并+融合救')

**🧪 胶囊练习**：实现 `min_read_time(num_bytes, tbps)`（流式读字节数的时间下界，秒）。这是任何访存受限内核的**理论最快时间**——你的内核再优化也快不过它。

In [ ]:
def min_read_time(num_bytes, tbps):
    # TODO: 返回 num_bytes / (tbps * 1e12)  秒
    raise NotImplementedError

In [ ]:
# 自测
t = min_read_time(4096 * 4096 * 2, 3.35)
assert abs(t - (4096 * 4096 * 2) / (3.35e12)) < 1e-15
# 带宽翻倍 -> 时间减半
assert abs(min_read_time(1e9, 2.0) / min_read_time(1e9, 4.0) - 2.0) < 1e-9
print(f'读 32MB @ 3.35TB/s 下界 = {t*1e6:.1f} us')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def min_read_time(num_bytes, tbps):
    return num_bytes / (tbps * 1e12)

---
## 🔧 旁注：对应的 Triton 内核长什么样

本模块的「合并访问」与「分块转置」，在 Triton 里是这样（伪代码，**本环境不跑**）：

```python
import triton, triton.language as tl

# (1) 合并访问：tl.arange 产生连续 offset，编译器自动合并成最少事务
@triton.jit
def scale_kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)   # 连续 -> 合并访问
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)       # 合并读
    tl.store(y_ptr + offs, x * 2.0, mask=mask) # 合并写

# (2) 分块转置：2D 块的 load/store。Triton 的内存合并与 bank 由编译器处理，
#     但分块结构(把一块搬上来再换方向写回)正是我们 numpy 模拟的 transpose_tiled。
@triton.jit
def transpose_kernel(in_ptr, out_ptr, H, W, BLK: tl.constexpr):
    pid_h, pid_w = tl.program_id(0), tl.program_id(1)
    rows = pid_h * BLK + tl.arange(0, BLK)
    cols = pid_w * BLK + tl.arange(0, BLK)
    in_idx  = rows[:, None] * W + cols[None, :]          # 读 A 的一块(按行连续)
    blk = tl.load(in_ptr + in_idx, mask=(rows[:,None]<H)&(cols[None,:]<W))
    out_idx = cols[:, None] * H + rows[None, :]          # 写 B 的对应块(按行连续)
    tl.store(out_ptr + out_idx, tl.trans(blk), mask=(cols[:,None]<W)&(rows[None,:]<H))
```

对应关系：`tl.arange` 的连续 offset ↔ 合并访问；2D 块 load/store + `tl.trans` ↔ 我们的 `tile_buf` 借道；Triton 把 cache-line 合并与 bank 冲突的细节自动处理掉——你只要把**分块结构**写对，正是本模块练的东西。

### 小结
- 内存是**金字塔**：register > shared/L1 > L2 > global(HBM)。性能 = 少碰 HBM、多在上层复用。
- **合并访问**：warp 的 32 个地址落在几条 cache line 上，就是几次事务；连续=1 次(100% 带宽)，跨步=32 次(3% 带宽)。
- **行优先布局** + 线程映射必须协同：按行=合并，按列=跨步；让 threadIdx 顺着内存变化最快的维走。
- **shared memory** 复用数据 + 线程协作，但写后读必须 `__syncthreads()`；列向访问当心 **bank 冲突**，**padding +1** 可消。
- **分块转置** = 合并读 + 片上转置 + 合并写 + padding 消冲突，是访存受限优化的集大成小例。
- **算子融合** = 不让中间结果落 HBM，与「分块复用」同源。

下一站：**模块 03 · 分块矩阵乘** —— 把「搬进 shared 复用」用到矩阵乘，把算术强度从 O(1) 提到 O(tile)。